# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codingsheep17/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#initializing the repo
import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/codingsheep17/flyrank-ml-internship.git

os.chdir("flyrank-ml-internship")
print(os.getcwd())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 229 (delta 112), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (229/229), 2.50 MiB | 16.21 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/content/flyrank-ml-internship


In [2]:
#installing the dataset
!pip install duckdb huggingface_hub -q

from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded:", "HF_TOKEN" in os.environ)

Token loaded: True


In [3]:
import duckdb

con = duckdb.connect()
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")
print("DuckDB secret configured")

DuckDB secret configured


In [5]:
dim_content_schema = con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' LIMIT 1
""").df()

print(dim_content_schema.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [6]:
staleness_check = con.sql("""
SELECT
    CASE
        WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 90 THEN 'fresh'
        WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 180 THEN 'aging'
        WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 365 THEN 'stale'
        ELSE 'very_stale'
    END as staleness_bucket,
    COUNT(*) as n,
    AVG(f.gsc_clicks) as avg_clicks
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d
ON f.content_hash_id = d.content_hash_id
WHERE f.gsc_data_available IS TRUE
GROUP BY staleness_bucket
ORDER BY avg_clicks DESC
""").df()

staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_clicks
0,fresh,3599822,0.227930
1,aging,9622,0.135003
2,stale,1617,0.016698


In [7]:
#now the ctr
ctr_position_check = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN 'top_3'
        WHEN gsc_avg_position <= 10 THEN 'page_1'
        WHEN gsc_avg_position <= 20 THEN 'page_2'
        ELSE 'deep'
    END as position_bucket,
    COUNT(*) as n,
    AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY position_bucket
ORDER BY avg_ctr DESC
""").df()

ctr_position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,top_3,727362,0.004756
1,page_1,1456122,0.003473
2,page_2,519223,0.002770
3,deep,908354,0.001289


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: Staleness — Verdict: CONFIRMED
Average clicks clearly decline as content gets staler: fresh pages average 0.228 clicks, aging pages 0.135, stale pages just 0.017 — roughly a 13x drop from fresh to stale. This confirms the signal behind FlyRank's refresh flags: staleness is genuinely associated with declining performance in this data. Note: no pages fell into "very_stale" (365+ days) in this slice — worth noting as a limitation of this month's sample.

Signal 2: CTR vs Position — Verdict: CONFIRMED
Average CTR clearly declines with worse position: top_3 pages average 0.48% CTR, dropping to 0.35% (page_1), 0.28% (page_2), and just 0.13% for deep results — roughly a 3.7x drop from top_3 to deep. This confirms the signal behind FlyRank's CTR-fix logic: a page's CTR is strongly tied to its ranking position, so any "low CTR" flag must account for position tier rather than comparing raw CTR across all pages equally.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.